In [5]:
import os
os.chdir("/home/sar_hydro/STUDIES/EtudesEB/PythonProject")

In [8]:
"""
À coller dans une cellule de notebook.
Affiche une carte Folium avec une station satellite (HW Next ou DAHITI)
et ses N stations in situ les plus proches, la plus proche étant
mise en évidence.
"""

import sqlite3
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point

# ═══════════════════════════════════════════════════════════════
# PARAMÈTRES — à modifier ici à chaque utilisation
# ═══════════════════════════════════════════════════════════════
SOURCE       = "dahiti"      # "dahiti" ou "hwnext"
STATION_CODE = "13386"       # code de la station satellite à inspecter
N_INSITU     = 1             # nombre de stations in situ les plus proches à afficher

HWNEXT_DB  = "./data/hydroweb_next.db"
DAHITI_DB  = "./data/dahiti.db"
INSITU_SHP = "./data/insitu/shp/station_schapi_alti_ref_2025_river.gpkg"

# Couleurs -- satellite en rouge, in situ en noir (dégradé de gris pour
# les rangs suivants si N_INSITU > 1), pour un meilleur contraste visuel
COLOR_SAT = "#C0392B"
COLOR_INSITU_CLOSEST = "#000000"

# ═══════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════
def get_coords_station(source, code):
    db_path = HWNEXT_DB if source == "hwnext" else DAHITI_DB
    conn = sqlite3.connect(db_path)
    for c in [str(code), str(code).zfill(13)]:
        df = pd.read_sql(
            "SELECT station_code, reference_longitude AS lon, reference_latitude AS lat "
            "FROM stations WHERE station_code = ?",
            conn, params=(c,)
        )
        if not df.empty:
            conn.close()
            return float(df.iloc[0]["lon"]), float(df.iloc[0]["lat"]), df.iloc[0]["station_code"]
    conn.close()
    return None, None, None


def get_n_insitu_proches(lon, lat, n=5):
    """Retourne un DataFrame des n stations in situ les plus proches, triées par distance."""
    pt = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs("EPSG:2154")[0]
    dist = gdf_insitu_proj.geometry.distance(pt) / 1000  # km
    idx_sorted = dist.sort_values().index[:n]

    rows = []
    for idx in idx_sorted:
        rows.append({
            "code_sta": gdf_insitu_proj.loc[idx, "code_sta"],
            "dist_km" : round(dist[idx], 2),
            "lon"     : gdf_insitu_wgs.loc[idx, "geometry"].x,
            "lat"     : gdf_insitu_wgs.loc[idx, "geometry"].y,
        })
    return pd.DataFrame(rows)


def couleur_rang(i):
    """Couleur dégradée (nuances de gris/noir) selon le rang de proximité
    (0 = plus proche, déjà géré séparément en noir plein)."""
    palette = ["#000000", "#424242", "#757575", "#9E9E9E", "#BDBDBD"]
    return palette[i] if i < len(palette) else "#CCCCCC"


# ═══════════════════════════════════════════════════════════════
# CHARGEMENT SHAPEFILE INSITU
# ═══════════════════════════════════════════════════════════════
gdf_insitu      = gpd.read_file(INSITU_SHP)
gdf_insitu_proj = gdf_insitu.to_crs("EPSG:2154")
gdf_insitu_wgs  = gdf_insitu.to_crs("EPSG:4326")

# ═══════════════════════════════════════════════════════════════
# RÉCUPÉRATION COORDONNÉES STATION + INSITU PROCHES
# ═══════════════════════════════════════════════════════════════
lon, lat, code_norm = get_coords_station(SOURCE, STATION_CODE)

if lon is None:
    print(f"⚠ Station {STATION_CODE} introuvable dans la base {SOURCE}")
else:
    df_proches = get_n_insitu_proches(lon, lat, n=N_INSITU)
    print(f"Station {SOURCE.upper()} {code_norm}  —  lon={lon:.4f}, lat={lat:.4f}\n")
    print(df_proches.to_string(index=False))

    # ── Carte ────────────────────────────────────────────────
    m = folium.Map(location=[lat, lon], zoom_start=10, tiles="OpenStreetMap")

    # Station satellite (alti) -- ROUGE
    folium.CircleMarker(
        location=[lat, lon],
        radius=10,
        color="white", weight=2,
        fill=True, fill_color=COLOR_SAT, fill_opacity=0.95,
        popup=folium.Popup(f"<b>Station {SOURCE.upper()} {code_norm}</b><br>"
                            f"lon/lat : {lon:.4f}, {lat:.4f}", max_width=250),
        tooltip=f"{SOURCE.upper()} {code_norm}",
    ).add_to(m)
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            html=f'<div style="font-size:10px;color:{COLOR_SAT};font-weight:bold;'
                 f'margin-left:13px;margin-top:-6px;">{code_norm}</div>'
        )
    ).add_to(m)

    # Stations in situ (les N plus proches) -- NOIR (dégradé de gris pour les rangs suivants)
    for i, row in df_proches.iterrows():
        is_closest = (i == 0)
        color = COLOR_INSITU_CLOSEST if is_closest else couleur_rang(i)
        radius = 9 if is_closest else 7

        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=radius,
            color="white", weight=2,
            fill=True, fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(
                f"<b>In situ {row['code_sta']}</b><br>"
                f"Rang : {i+1}{'  (plus proche)' if is_closest else ''}<br>"
                f"Distance : {row['dist_km']} km",
                max_width=250),
            tooltip=f"{row['code_sta']} — {row['dist_km']} km"
                     + ("  ★ plus proche" if is_closest else ""),
        ).add_to(m)

        folium.Marker(
            location=[row["lat"], row["lon"]],
            icon=folium.DivIcon(
                html=f'<div style="font-size:9px;color:{color};font-weight:bold;'
                     f'margin-left:11px;margin-top:-6px;">{row["code_sta"]}</div>'
            )
        ).add_to(m)

    # ── Légende (agrandie) ───────────────────────────────────
    legende = f"""
    <div style="position:fixed;bottom:25px;left:25px;z-index:1000;
                background:white;padding:16px 22px;border-radius:8px;
                box-shadow:0 2px 8px rgba(0,0,0,0.4);font-size:16px;
                line-height:1.9;min-width:260px;">
      <b style="font-size:17px;">{SOURCE.upper()} {code_norm}</b><br>
      <span style="color:{COLOR_SAT};font-size:22px;">●</span> Station satellite (alti)<br>
      <span style="color:{COLOR_INSITU_CLOSEST};font-size:22px;">●</span> In situ le plus proche ({df_proches.iloc[0]['dist_km']} km)<br>
      <span style="color:#9E9E9E;font-size:22px;">●</span> Autres in situ proches ({N_INSITU-1} suivants)
    </div>
    """
    m.get_root().html.add_child(folium.Element(legende))

    display(m)

Station DAHITI 0000000013386  —  lon=-0.3832, lat=44.6880

  code_sta  dist_km       lon       lat
O960001001     7.47 -0.324223 44.635551


In [ ]:
"""
carte_multi_alti_insitu.py
════════════════════════════════════════════════════════════════════════
À coller dans une cellule de notebook.

Sélectionne un maximum de stations alti (HW Next ou DAHITI) tout en
garantissant qu'aucune paire de stations sélectionnées ne soit à moins
de ALTI_MIN_DIST_KM l'une de l'autre (sélection gloutonne, priorité aux
stations avec le plus de mesures alti).

Pour chaque station alti sélectionnée, affiche sur une carte Folium
unique (un calque par station, activable via le contrôle en haut à
droite) :
  - la station alti (bleu)
  - l'insitu le plus proche en distance (vert)
  - l'insitu avec le meilleur NSE (insitu vs alti BRUTE) (orange)
  - si le plus proche EST le meilleur NSE : marqueur dédié (violet)
  - les autres insitu candidats dans le rayon INSITU_DIST_MAX_KM (gris)

Aucun modèle ici : l'alti vient directement de la table `measurements`
(orthometric_height, is_valid=1).
════════════════════════════════════════════════════════════════════════
"""

import sqlite3
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import Point

# ═══════════════════════════════════════════════════════════════
# PARAMÈTRES — à modifier ici à chaque utilisation
# ═══════════════════════════════════════════════════════════════
SOURCE             = "hwnext"   # "hwnext" ou "dahiti"

ALTI_MIN_DIST_KM   = 10.0       # distance minimale entre 2 stations alti sélectionnées
INSITU_DIST_MAX_KM = 10.0       # rayon de recherche des insitu autour de chaque station alti
MAX_STATIONS       = 100        # nombre max de stations alti à afficher (None = pas de limite)

WINDOW_DAYS = 3                 # tolérance (jours) pour aligner une mesure insitu sur une date alti
MIN_PAIRS   = 20                # nb minimal de paires alti/insitu pour valider un candidat insitu

DATE_MIN = "2016-01-01"
DATE_MAX = "2025-12-31"

HWNEXT_DB  = "./data/hydroweb_next.db"
DAHITI_DB  = "./data/dahiti.db"
INSITU_DB  = "./data/insitu_data.db"
INSITU_SHP = "./data/insitu/shp/station_schapi_alti_ref_2025_river.gpkg"

SAT_DB = HWNEXT_DB if SOURCE == "hwnext" else DAHITI_DB

# ═══════════════════════════════════════════════════════════════
# HELPERS — calcul NSE
# ═══════════════════════════════════════════════════════════════
def zscore(arr):
    arr = np.asarray(arr, dtype=float)
    m = ~np.isnan(arr)
    if m.sum() < 2:
        return arr * np.nan
    mu, sig = arr[m].mean(), arr[m].std()
    return (arr - mu) / sig if sig > 0 else arr * 0

def nse(obs, sim):
    m = ~(np.isnan(obs) | np.isnan(sim))
    if m.sum() < 5:
        return np.nan
    o, s = obs[m], sim[m]
    d = np.sum((o - o.mean()) ** 2)
    return float(1 - np.sum((o - s) ** 2) / d) if d > 0 else np.nan

def align_insitu(dates, df_ins, window_days):
    wl  = np.full(len(dates), np.nan)
    id_ = np.array(df_ins["date"].values, dtype="datetime64[D]")
    iv  = df_ins["wl"].values
    for i, d in enumerate(np.array(dates, dtype="datetime64[D]")):
        diff = np.abs((id_ - d).astype(float))
        idx  = int(np.argmin(diff))
        if diff[idx] <= window_days:
            wl[i] = iv[idx]
    return wl

# ═══════════════════════════════════════════════════════════════
# HELPERS — accès données alti / insitu
# ═══════════════════════════════════════════════════════════════
def get_all_stations(source):
    db_path = HWNEXT_DB if source == "hwnext" else DAHITI_DB
    conn = sqlite3.connect(db_path)
    df = pd.read_sql("""
        SELECT station_code, reference_longitude AS lon, reference_latitude AS lat,
               nb_measurements, river_name
        FROM stations
        WHERE reference_longitude IS NOT NULL AND reference_latitude IS NOT NULL
    """, conn)
    conn.close()
    df["nb_measurements"] = df["nb_measurements"].fillna(0)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    return gdf

def get_alti_series(source, code_norm):
    db_path = HWNEXT_DB if source == "hwnext" else DAHITI_DB
    conn = sqlite3.connect(db_path)
    df = pd.read_sql("""
        SELECT measure_date AS date, orthometric_height AS wl
        FROM measurements
        WHERE station_code = ? AND is_valid = 1
          AND measure_date >= ? AND measure_date <= ?
        ORDER BY measure_date
    """, conn, params=(code_norm, DATE_MIN, DATE_MAX))
    conn.close()
    df["date"] = pd.to_datetime(df["date"])
    return df.dropna(subset=["wl"])

def get_insitu_series(code_sta):
    conn = sqlite3.connect(INSITU_DB)
    df = pd.read_sql("""
        SELECT date, h_med_wsh AS wl FROM mesures_insitu
        WHERE code_sta = ? AND date >= ? AND date <= ?
        ORDER BY date
    """, conn, params=(code_sta, DATE_MIN, DATE_MAX))
    conn.close()
    df["date"] = pd.to_datetime(df["date"])
    df = df.dropna(subset=["wl"])
    return df if len(df) >= 5 else None

def get_insitu_candidats(lon, lat, dist_max_km, gdf_proj):
    pt   = gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326").to_crs("EPSG:2154")[0]
    dist = gdf_proj.geometry.distance(pt) / 1000
    candidats = dist[dist <= dist_max_km].sort_values()
    return [(gdf_proj.loc[idx, "code_sta"], d) for idx, d in candidats.items()]

# ═══════════════════════════════════════════════════════════════
# SÉLECTION GLOUTONNE DES STATIONS ALTI ESPACÉES
# ═══════════════════════════════════════════════════════════════
def select_spaced_stations(gdf_stations_proj, min_dist_km, max_stations=None):
    """Trie par nb_measurements décroissant, ajoute gloutonnement si distance
    >= min_dist_km à toutes les stations déjà sélectionnées."""
    gdf_sorted = gdf_stations_proj.sort_values("nb_measurements", ascending=False).reset_index(drop=True)
    selected_rows  = []
    selected_geoms = []
    for _, row in gdf_sorted.iterrows():
        geom = row.geometry
        if all(geom.distance(g) / 1000 >= min_dist_km for g in selected_geoms):
            selected_rows.append(row)
            selected_geoms.append(geom)
            if max_stations and len(selected_rows) >= max_stations:
                break
    return gpd.GeoDataFrame(selected_rows, crs=gdf_stations_proj.crs).reset_index(drop=True)

# ═══════════════════════════════════════════════════════════════
# CHARGEMENT
# ═══════════════════════════════════════════════════════════════
print(f"Source : {SOURCE.upper()}  |  Espacement alti mini : {ALTI_MIN_DIST_KM} km  "
      f"|  Rayon insitu : {INSITU_DIST_MAX_KM} km\n")

gdf_insitu      = gpd.read_file(INSITU_SHP)
gdf_insitu_proj = gdf_insitu.to_crs("EPSG:2154")
gdf_insitu_wgs  = gdf_insitu.to_crs("EPSG:4326")

gdf_alti_all  = get_all_stations(SOURCE)
gdf_alti_proj = gdf_alti_all.to_crs("EPSG:2154")

gdf_alti_sel = select_spaced_stations(gdf_alti_proj, ALTI_MIN_DIST_KM, MAX_STATIONS)
gdf_alti_sel_wgs = gdf_alti_sel.to_crs("EPSG:4326")

print(f"{len(gdf_alti_sel)} stations alti sélectionnées sur {len(gdf_alti_all)} disponibles "
      f"(espacement >= {ALTI_MIN_DIST_KM} km)\n")
print(gdf_alti_sel[["station_code", "nb_measurements", "river_name"]].to_string(index=False))

# ═══════════════════════════════════════════════════════════════
# POUR CHAQUE STATION ALTI SÉLECTIONNÉE : ANALYSE INSITU
# ═══════════════════════════════════════════════════════════════
COLOR_SAT      = "#1565C0"  # bleu   - station alti
COLOR_CLOSEST  = "#2E7D32"  # vert   - plus proche
COLOR_BESTNSE  = "#FB8C00"  # orange - meilleur NSE
COLOR_BOTH     = "#8E24AA"  # violet - plus proche ET meilleur NSE
COLOR_OTHER    = "#9E9E9E"  # gris   - autres candidats

lat0 = gdf_alti_sel_wgs.geometry.y.mean()
lon0 = gdf_alti_sel_wgs.geometry.x.mean()
m = folium.Map(location=[lat0, lon0], zoom_start=7, tiles="OpenStreetMap")

summary_rows = []

for _, sta in gdf_alti_sel_wgs.iterrows():
    code_norm = sta["station_code"]
    lon, lat  = sta.geometry.x, sta.geometry.y

    fg = folium.FeatureGroup(name=f"{code_norm} ({sta['river_name'] or ''})", show=True)

    # Station alti
    fg.add_child(folium.CircleMarker(
        location=[lat, lon], radius=9, color="white", weight=2,
        fill=True, fill_color=COLOR_SAT, fill_opacity=0.95,
        popup=folium.Popup(f"<b>{SOURCE.upper()} {code_norm}</b><br>"
                            f"Rivière : {sta['river_name']}<br>"
                            f"lon/lat : {lon:.4f}, {lat:.4f}", max_width=250),
        tooltip=f"{SOURCE.upper()} {code_norm}",
    ))
    fg.add_child(folium.Circle(
        location=[lat, lon], radius=INSITU_DIST_MAX_KM * 1000,
        color=COLOR_SAT, weight=1, fill=False, dash_array="5,5",
    ))

    df_alti = get_alti_series(SOURCE, code_norm)
    if df_alti.empty:
        print(f"⚠ {code_norm} : aucune mesure alti valide -> pas d'analyse insitu")
        m.add_child(fg)
        continue

    obs_z = zscore(df_alti["wl"].values)
    candidats = get_insitu_candidats(lon, lat, INSITU_DIST_MAX_KM, gdf_insitu_proj)

    cand_rows = []
    for code_ins, dist_km in candidats:
        df_ins = get_insitu_series(code_ins)
        if df_ins is None:
            continue
        ins_wl  = align_insitu(df_alti["date"].values, df_ins, WINDOW_DAYS)
        n_pairs = int(np.sum(~np.isnan(ins_wl)))
        if n_pairs < MIN_PAIRS:
            continue
        ins_z = zscore(ins_wl)
        nse_val = nse(ins_z, obs_z)
        row_ins = gdf_insitu_wgs[gdf_insitu_wgs["code_sta"] == code_ins].iloc[0]
        cand_rows.append({
            "code_sta": code_ins, "dist_km": round(dist_km, 2),
            "n_pairs": n_pairs, "nse": nse_val,
            "lon": row_ins.geometry.x, "lat": row_ins.geometry.y,
        })

    if not cand_rows:
        print(f"⚠ {code_norm} : aucun insitu valide dans {INSITU_DIST_MAX_KM} km")
        m.add_child(fg)
        summary_rows.append({"station": code_norm, "n_insitu_valides": 0,
                              "insitu_proche": None, "insitu_meilleur_nse": None, "nse": np.nan})
        continue

    df_cand = pd.DataFrame(cand_rows).sort_values("dist_km").reset_index(drop=True)
    code_closest = df_cand.iloc[0]["code_sta"]
    df_valid = df_cand.dropna(subset=["nse"])
    code_best_nse = df_valid.loc[df_valid["nse"].idxmax(), "code_sta"] if not df_valid.empty else None

    for _, ins_row in df_cand.iterrows():
        is_closest = ins_row["code_sta"] == code_closest
        is_best    = ins_row["code_sta"] == code_best_nse

        if is_closest and is_best:
            color, radius, label = COLOR_BOTH, 10, "plus proche + meilleur NSE"
        elif is_closest:
            color, radius, label = COLOR_CLOSEST, 9, "plus proche"
        elif is_best:
            color, radius, label = COLOR_BESTNSE, 9, "meilleur NSE"
        else:
            color, radius, label = COLOR_OTHER, 6, "autre candidat"

        nse_txt = f"{ins_row['nse']:.3f}" if pd.notna(ins_row["nse"]) else "N/A"

        fg.add_child(folium.CircleMarker(
            location=[ins_row["lat"], ins_row["lon"]], radius=radius, color="white", weight=2,
            fill=True, fill_color=color, fill_opacity=0.9,
            popup=folium.Popup(
                f"<b>Insitu {ins_row['code_sta']}</b> ({label})<br>"
                f"Lié à alti {code_norm}<br>"
                f"Distance : {ins_row['dist_km']} km<br>"
                f"NSE vs alti brute : {nse_txt}<br>"
                f"N paires : {ins_row['n_pairs']}",
                max_width=260),
            tooltip=f"{ins_row['code_sta']} — {label} — NSE={nse_txt}",
        ))
        fg.add_child(folium.PolyLine(
            locations=[[lat, lon], [ins_row["lat"], ins_row["lon"]]],
            color=color, weight=2.2 if (is_closest or is_best) else 1,
            opacity=0.85 if (is_closest or is_best) else 0.4,
            dash_array=None if (is_closest or is_best) else "4,4",
        ))

    m.add_child(fg)

    best_nse_val = df_valid["nse"].max() if not df_valid.empty else np.nan
    summary_rows.append({
        "station": code_norm,
        "n_insitu_valides": len(df_cand),
        "insitu_proche": code_closest,
        "insitu_meilleur_nse": code_best_nse,
        "nse": round(best_nse_val, 3) if pd.notna(best_nse_val) else np.nan,
    })

# Légende fixe + contrôle de calques
legende = f"""
<div style="position:fixed;bottom:20px;left:20px;z-index:1000;
            background:white;padding:10px 14px;border-radius:6px;
            box-shadow:0 1px 5px rgba(0,0,0,0.4);font-size:12px;">
  <b>{SOURCE.upper()} — {len(gdf_alti_sel)} stations (espacement >= {ALTI_MIN_DIST_KM} km)</b><br>
  <span style="color:{COLOR_SAT};">●</span> Station alti<br>
  <span style="color:{COLOR_CLOSEST};">●</span> Insitu plus proche<br>
  <span style="color:{COLOR_BESTNSE};">●</span> Insitu meilleur NSE<br>
  <span style="color:{COLOR_BOTH};">●</span> Plus proche = meilleur NSE<br>
  <span style="color:{COLOR_OTHER};">●</span> Autres candidats (rayon {INSITU_DIST_MAX_KM} km)
</div>
"""
m.get_root().html.add_child(folium.Element(legende))
folium.LayerControl(collapsed=False).add_to(m)

print(f"\n{'='*70}")
print(pd.DataFrame(summary_rows).to_string(index=False))

display(m)

In [ ]:
# Identifier le 2e candidat (~9.99 km) et voir pourquoi il a été exclu
candidats_debug = get_insitu_candidats(lon, lat, DIST_MAX_KM, gdf_insitu_proj)
print(candidats_debug)

for code_ins, dist_km in candidats_debug:
    df_ins = get_insitu_series(code_ins)
    if df_ins is None:
        print(f"{code_ins} ({dist_km:.2f} km) -> exclu : moins de 5 mesures insitu au total")
        continue
    ins_wl = align_insitu(df_alti["date"].values, df_ins, WINDOW_DAYS)
    n_pairs = int(np.sum(~np.isnan(ins_wl)))
    print(f"{code_ins} ({dist_km:.2f} km) -> {len(df_ins)} mesures insitu au total, {n_pairs} paires alignées (seuil MIN_PAIRS={MIN_PAIRS})")